In [1]:
import warnings
import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ---------------------------------------------------------
# 1. LOAD AND VALIDATE DATA
# ---------------------------------------------------------
df = pd.read_csv("merged_data.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

# Check duplicate monthly dates
if df["Date"].duplicated().any():
    raise ValueError("Duplicate monthly dates were found.")

series = (
    df.set_index("Date")["E&E Exports"]
      .astype(float)
      .asfreq("MS")
)

if series.isna().any():
    raise ValueError(
        "Missing months or missing E&E export values were found."
    )

print(
    "Complete series:",
    len(series),
    "|",
    series.index.min(),
    "to",
    series.index.max()
)

# ---------------------------------------------------------
# 2. DEVELOPMENT SAMPLE AND 80:20 SPLIT
# January 1990–December 2023 = 408 observations
# ---------------------------------------------------------
development = series.loc[:"2023-12-01"]

split_index = int(len(development) * 0.80)

train_dev = development.iloc[:split_index]
test_dev = development.iloc[split_index:]

print("\nDevelopment observations:", len(development))
print(
    "Initial training partition:",
    len(train_dev),
    "|",
    train_dev.index.min(),
    "to",
    train_dev.index.max()
)
print(
    "Historical test partition:",
    len(test_dev),
    "|",
    test_dev.index.min(),
    "to",
    test_dev.index.max()
)

# Expected:
# Development = 408
# Train = 326
# Test = 82

# ---------------------------------------------------------
# 3. STATIONARITY TESTING ON INITIAL TRAINING PARTITION
# ---------------------------------------------------------
adf_raw = adfuller(train_dev)

print("\nADF test: raw training series")
print(f"Statistic: {adf_raw[0]:.4f}")
print(f"p-value:   {adf_raw[1]:.6f}")

diff_train = train_dev.diff().dropna()
adf_diff = adfuller(diff_train)

print("\nADF test: first-differenced training series")
print(f"Statistic: {adf_diff[0]:.4f}")
print(f"p-value:   {adf_diff[1]:.6f}")

# ---------------------------------------------------------
# 4. ORDER SELECTION ON INITIAL TRAINING PARTITION
# ---------------------------------------------------------
candidate_orders = [
    (1, 1, 0),
    (1, 1, 1),
    (2, 1, 0),
    (2, 1, 1),
    (2, 1, 2),
    (3, 1, 1)
]

order_results = []

for order in candidate_orders:
    try:
        model = ARIMA(
            train_dev,
            order=order,
            enforce_stationarity=True,
            enforce_invertibility=True
        ).fit()

        order_results.append({
            "Order": order,
            "AIC": model.aic,
            "BIC": model.bic,
            "Converged": model.mle_retvals.get("converged", np.nan)
        })

    except Exception as error:
        order_results.append({
            "Order": order,
            "AIC": np.nan,
            "BIC": np.nan,
            "Converged": False
        })
        print(f"Could not fit ARIMA{order}: {error}")

order_selection_df = (
    pd.DataFrame(order_results)
      .dropna(subset=["AIC"])
      .sort_values("AIC")
      .reset_index(drop=True)
)

print("\nOrder-selection results:")
print(order_selection_df)

best_order = tuple(order_selection_df.iloc[0]["Order"])
print("\nSelected order:", best_order)

order_selection_df.to_csv(
    "arima_order_selection_development.csv",
    index=False
)

# ---------------------------------------------------------
# 5. FIT DEVELOPMENT MODEL AND CHECK RESIDUALS
# ---------------------------------------------------------
development_model = ARIMA(
    train_dev,
    order=best_order,
    enforce_stationarity=True,
    enforce_invertibility=True
).fit()

print("\nDevelopment model")
print("AIC:", development_model.aic)
print("BIC:", development_model.bic)

# Remove initial residuals affected by likelihood initialisation
burn = development_model.loglikelihood_burn
development_residuals = development_model.resid.iloc[burn:]

# Adjustment for the number of estimated AR and MA parameters
model_df = best_order[0] + best_order[2]

lb_development = acorr_ljungbox(
    development_residuals,
    lags=[10, 20],
    model_df=model_df,
    return_df=True
)

print("\nDevelopment-model Ljung-Box test:")
print(lb_development)

lb_development.to_csv(
    "arima_ljung_box_development.csv",
    index=True
)

# ---------------------------------------------------------
# 6. WALK-FORWARD, EXPANDING-WINDOW EVALUATION
# ---------------------------------------------------------
history = train_dev.copy()
walk_forward_records = []

for origin_number in range(len(test_dev)):

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        fitted_model = ARIMA(
            history,
            order=best_order,
            enforce_stationarity=True,
            enforce_invertibility=True
        ).fit()

    forecast_result = fitted_model.get_forecast(steps=3)
    forecasts = forecast_result.predicted_mean

    # Record forecasts for horizons 1, 2 and 3 where actuals exist
    for horizon in range(1, 4):
        target_position = origin_number + horizon - 1

        if target_position < len(test_dev):
            target_date = test_dev.index[target_position]
            actual_value = test_dev.iloc[target_position]
            forecast_value = forecasts.iloc[horizon - 1]

            walk_forward_records.append({
                "Origin Date": history.index[-1],
                "Target Date": target_date,
                "Horizon": horizon,
                "Actual": actual_value,
                "Forecast": forecast_value,
                "Error": actual_value - forecast_value
            })

    # Reveal only the next actual observation
    next_actual = test_dev.iloc[[origin_number]]
    history = pd.concat([history, next_actual])

walk_forward_df = pd.DataFrame(walk_forward_records)

print("\nWalk-forward forecast sample:")
print(walk_forward_df.head())

# ---------------------------------------------------------
# 7. HORIZON-SPECIFIC TEST METRICS
# ---------------------------------------------------------
performance_records = []

for horizon in [1, 2, 3]:
    horizon_data = walk_forward_df[
        walk_forward_df["Horizon"] == horizon
    ]

    actual = horizon_data["Actual"]
    predicted = horizon_data["Forecast"]

    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, predicted)

    performance_records.append({
        "Forecast Horizon": f"{horizon}-month",
        "Number of Forecasts": len(horizon_data),
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

performance_df = pd.DataFrame(performance_records)

print("\nWalk-forward test-period performance:")
print(performance_df)

walk_forward_df.to_csv(
    "arima_walk_forward_forecasts.csv",
    index=False
)

performance_df.to_csv(
    "arima_model_performance.csv",
    index=False
)

# ---------------------------------------------------------
# 8. FINAL MODEL THROUGH DECEMBER 2024
# ---------------------------------------------------------
final_training = series.loc[:"2024-12-01"]

print(
    "\nFinal operational training sample:",
    len(final_training),
    "|",
    final_training.index.min(),
    "to",
    final_training.index.max()
)

# Refit the six candidates through December 2024
final_order_results = []

for order in candidate_orders:
    try:
        model = ARIMA(
            final_training,
            order=order,
            enforce_stationarity=True,
            enforce_invertibility=True
        ).fit()

        final_order_results.append({
            "Order": order,
            "AIC": model.aic,
            "BIC": model.bic,
            "Converged": model.mle_retvals.get("converged", np.nan)
        })

    except Exception as error:
        print(f"Final ARIMA{order} failed: {error}")

final_order_selection_df = (
    pd.DataFrame(final_order_results)
      .sort_values("AIC")
      .reset_index(drop=True)
)

print("\nFinal-sample order-selection results:")
print(final_order_selection_df)

final_order_selection_df.to_csv(
    "arima_order_selection_final.csv",
    index=False
)

# Keep the originally selected specification for final deployment
final_model = ARIMA(
    final_training,
    order=best_order,
    enforce_stationarity=True,
    enforce_invertibility=True
).fit()

print("\nFinal operational model:", best_order)
print("AIC:", final_model.aic)
print("BIC:", final_model.bic)

# Final-model residual diagnostic
burn_final = final_model.loglikelihood_burn
final_residuals = final_model.resid.iloc[burn_final:]

lb_final = acorr_ljungbox(
    final_residuals,
    lags=[10, 20],
    model_df=model_df,
    return_df=True
)

print("\nFinal-model Ljung-Box test:")
print(lb_final)

lb_final.to_csv(
    "arima_ljung_box_final.csv",
    index=True
)

# ---------------------------------------------------------
# 9. FORECAST JANUARY–MARCH 2025
# ---------------------------------------------------------
forecast_result = final_model.get_forecast(steps=3)

forecast_mean = forecast_result.predicted_mean
forecast_interval = forecast_result.conf_int(alpha=0.05)

actual_q1 = series.reindex(forecast_mean.index)

if actual_q1.isna().any():
    raise ValueError(
        "Actual January–March 2025 observations are not aligned "
        "with the forecast dates."
    )

comparison = pd.DataFrame({
    "Date": forecast_mean.index,
    "Actual": actual_q1.values,
    "Forecast": forecast_mean.values,
    "Lower 95% forecast interval": forecast_interval.iloc[:, 0].values,
    "Upper 95% forecast interval": forecast_interval.iloc[:, 1].values
})

comparison["Error"] = comparison["Actual"] - comparison["Forecast"]
comparison["Absolute Error"] = comparison["Error"].abs()
comparison["Absolute Percentage Error"] = (
    comparison["Absolute Error"] / comparison["Actual"]
) * 100

comparison["Covered by 95% interval"] = (
    (
        comparison["Actual"] >=
        comparison["Lower 95% forecast interval"]
    )
    &
    (
        comparison["Actual"] <=
        comparison["Upper 95% forecast interval"]
    )
)

comparison["Forecast Horizon"] = [
    "1-month",
    "2-month",
    "3-month"
]

print("\nJanuary–March 2025 validation:")
print(comparison)

comparison.to_csv(
    "arima_forecast_data.csv",
    index=False
)

# ---------------------------------------------------------
# 10. OVERALL Q1 2025 VALIDATION METRICS
# ---------------------------------------------------------
q1_mae = mean_absolute_error(
    comparison["Actual"],
    comparison["Forecast"]
)

q1_mse = mean_squared_error(
    comparison["Actual"],
    comparison["Forecast"]
)

q1_rmse = np.sqrt(q1_mse)

# R² can be calculated, but n=3 makes it unstable
q1_r2 = r2_score(
    comparison["Actual"],
    comparison["Forecast"]
)

q1_metrics_df = pd.DataFrame([{
    "Evaluation Period": "January–March 2025",
    "Number of Observations": len(comparison),
    "MAE": q1_mae,
    "MSE": q1_mse,
    "RMSE": q1_rmse,
    "R2": q1_r2
}])

print("\nOverall January–March 2025 metrics:")
print(q1_metrics_df)

q1_metrics_df.to_csv(
    "arima_q1_2025_validation_metrics.csv",
    index=False
)

print("\nFiles saved successfully.")

Complete series: 423 | 1990-01-01 00:00:00 to 2025-03-01 00:00:00

Development observations: 408
Initial training partition: 326 | 1990-01-01 00:00:00 to 2017-02-01 00:00:00
Historical test partition: 82 | 2017-03-01 00:00:00 to 2023-12-01 00:00:00

ADF test: raw training series
Statistic: 0.3140
p-value:   0.977979

ADF test: first-differenced training series
Statistic: -6.7915
p-value:   0.000000


/tmp/ipykernel_3492/1933771527.py:83: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_raw = adfuller(train_dev)
/tmp/ipykernel_3492/1933771527.py:90: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_diff = adfuller(diff_train)



Order-selection results:
       Order          AIC          BIC  Converged
0  (2, 1, 2)  5289.814057  5308.733183       True
1  (2, 1, 1)  5293.259274  5308.394575       True
2  (3, 1, 1)  5293.501135  5312.420261       True
3  (2, 1, 0)  5293.895949  5305.247425       True
4  (1, 1, 1)  5302.300756  5313.652232       True
5  (1, 1, 0)  5313.634272  5321.201923       True

Selected order: (2, 1, 2)

Development model
AIC: 5289.814056899344
BIC: 5308.733182810993

Development-model Ljung-Box test:
      lb_stat     lb_pvalue
10  30.702834  2.888460e-05
20  92.080277  1.033457e-12

Walk-forward forecast sample:
  Origin Date Target Date  Horizon        Actual      Forecast        Error
0  2017-02-01  2017-03-01        1  19723.905053  17188.799584  2535.105469
1  2017-02-01  2017-04-01        2  18022.401007  16681.797918  1340.603089
2  2017-02-01  2017-05-01        3  19977.778692  16825.914054  3151.864638
3  2017-03-01  2017-04-01        1  18022.401007  18032.922979   -10.521972
4 